# Semantic Search Testing

Test the vector retrieval quality by performing semantic searches against the ingested documents in pgvector.

## 1. Load Environment

In [ ]:
import os

import httpx
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

print("✅ Environment loaded")

## 2. Define Search Function

In [ ]:
import psycopg2
from openai import OpenAI

embedding_client = OpenAI(
    base_url=os.getenv("EMBEDDING_BASE_URL", "http://localhost:8000/v1"),
    api_key=os.getenv("EMBEDDING_API_KEY", "not-needed"),
    http_client=_http_client,
)


def semantic_search(query: str, top_k: int = 5):
    """Search pgvector for semantically similar chunks."""
    # Get query embedding
    resp = embedding_client.embeddings.create(
        input=[query],
        model=os.getenv("EMBEDDING_MODEL", "granite-embedding-278m-multilingual"),
    )
    query_embedding = resp.data[0].embedding

    # Search pgvector
    conn = psycopg2.connect(
        host=os.getenv("PGVECTOR_HOST", "localhost"),
        port=os.getenv("PGVECTOR_PORT", "5432"),
        dbname=os.getenv("PGVECTOR_DB", "doc_research"),
        user=os.getenv("PGVECTOR_USER", "postgres"),
        password=os.getenv("PGVECTOR_PASSWORD", "postgres"),
    )
    cur = conn.cursor()
    cur.execute(
        """SELECT document_name, chunk_index, content,
                  1 - (embedding <=> %s::vector) as similarity
           FROM document_chunks
           ORDER BY embedding <=> %s::vector
           LIMIT %s""",
        (str(query_embedding), str(query_embedding), top_k),
    )
    results = cur.fetchall()
    cur.close()
    conn.close()
    return results


print("✅ Search function defined")

## 3. Test Semantic Search

Run sample queries to verify retrieval quality.

In [ ]:
test_queries = [
    "What is the main architecture of this system?",
    "How does document parsing work?",
    "What are the key results and findings?",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    results = semantic_search(query, top_k=3)
    for i, (doc_name, chunk_idx, content, similarity) in enumerate(results):
        print(f"\n  [{i+1}] {doc_name} (chunk {chunk_idx}) — similarity: {similarity:.4f}")
        print(f"      {content[:150]}...")

print("\n✅ Semantic search test complete")

## 4. Evaluate Retrieval Quality

Check similarity score distribution to assess embedding quality.

In [ ]:
# Run broader search to check score distribution
results = semantic_search("document processing and analysis", top_k=10)

similarities = [r[3] for r in results]
print(f"Top-10 similarity scores:")
print(f"  Max:  {max(similarities):.4f}")
print(f"  Min:  {min(similarities):.4f}")
print(f"  Mean: {sum(similarities)/len(similarities):.4f}")
print(f"  Spread: {max(similarities) - min(similarities):.4f}")

if max(similarities) > 0.5:
    print("\n✅ Good retrieval quality — high similarity scores")
else:
    print("\n⚠️ Low similarity scores — may need better embeddings or more documents")